# SCC: OpenMP vs CUDA Benchmark

Benchmarks on large graphs. Runs all 3 methods and shows combined OMP vs CUDA comparison.

**Requires GPU runtime:** Runtime → Change runtime type → T4 GPU

In [ ]:
# @title 1. Mount Google Drive (for cached datasets)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 2. Clone repo, clean pull, check GPU
import os, sys, subprocess, time, re, urllib.request, gzip

REPO_URL = "https://github.com/ShashwaTTrigunayaT/DynamicGraphs_SCC.git"
REPO_DIR = "/content/DynamicGraphs_SCC"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin 2>&1 | tail -3
    !git reset --hard origin/main 2>&1 | tail -3
%cd {REPO_DIR}

!echo "=== GPU Check ==="
!nvidia-smi 2>&1 | head -12
!echo "=== CUDA ==="
!nvcc --version 2>&1 | tail -3
!echo "=== CPUs ==="
!nproc

In [ ]:
# @title 3. Auto-detect CUDA path, patch arch, build

try:
    cuda_candidates = sorted([d for d in os.listdir('/usr/local') if d.startswith('cuda')])
except FileNotFoundError:
    cuda_candidates = []
if cuda_candidates:
    cuda_home = f"/usr/local/{cuda_candidates[-1]}"
else:
    import shutil
    nvcc_path = shutil.which('nvcc')
    cuda_home = os.path.dirname(os.path.dirname(nvcc_path)) if nvcc_path else "/usr/local/cuda"
print(f"CUDA_HOME = {cuda_home}")

gpu_arch = "sm_75"
gpu_info = !nvidia-smi --query-gpu=name --format=csv,noheader 2>&1
name = gpu_info[0] if gpu_info else ""
if 'A100' in name: gpu_arch = "sm_80"
elif 'V100' in name: gpu_arch = "sm_70"
elif 'T4' in name: gpu_arch = "sm_75"
elif 'P100' in name: gpu_arch = "sm_60"
elif 'K80' in name: gpu_arch = "sm_37"
print(f"GPU: {name.strip()} → -arch={gpu_arch}")

with open("src_CUDA/Makefile", "r") as f:
    content = f.read()
content = content.replace("CUDA_HOME ?= /usr/local/cuda", f"CUDA_HOME ?= {cuda_home}")
for old in ["sm_70", "sm_75", "sm_80", "sm_60", "sm_37"]:
    content = content.replace(f"-arch={old}", f"-arch={gpu_arch}")
with open("src_CUDA/Makefile", "w") as f:
    f.write(content)
print("✓ Makefile patched")

print("\n=== Building OpenMP ===")
%cd /content/DynamicGraphs_SCC/src
!make -s clean 2>&1; make 2>&1 | tail -5
assert os.path.exists("../scc"), "OpenMP build failed!"
print("✓ OMP built")

print("\n=== Building CUDA ===")
%cd /content/DynamicGraphs_SCC/src_CUDA
!make -s clean 2>&1; make 2>&1 | tail -10
assert os.path.exists("scc_cuda"), "CUDA build failed!"
print("✓ CUDA built")

BINARY_OMP  = "/content/DynamicGraphs_SCC/scc"
BINARY_CUDA = "/content/DynamicGraphs_SCC/src_CUDA/scc_cuda"

In [ ]:
# @title 4. Load graphs (from Drive or download)

datasets_dir = "/content/large_graphs"
os.makedirs(datasets_dir, exist_ok=True)
graph_info = []

DRIVE_PATHS = [
    "/content/drive/MyDrive/colab_data/soc-pokec-relationships.txt",
    "/content/drive/MyDrive/soc-pokec-relationships.txt",
]

def count_edges(path):
    max_v = 0; count = 0
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                u, v = int(parts[0]), int(parts[1])
                max_v = max(max_v, u, v); count += 1
    return max_v, count

def get_local_graph(name, drive_paths, datasets_dir):
    graph_dir = os.path.join(datasets_dir, name)
    os.makedirs(graph_dir, exist_ok=True)
    out_path = os.path.join(graph_dir, "refined_edges.txt")
    if os.path.exists(out_path):
        n, m = count_edges(out_path)
        print(f"  {name}: cached ({n} nodes, {m} edges)")
        return name, n, m, name, out_path
    for dp in drive_paths:
        if os.path.exists(dp):
            print(f"  {name}: found in Drive, converting...")
            with open(dp) as f_in, open(out_path, "w") as f_out:
                for line in f_in:
                    f_out.write(line.replace("\t", " "))
            n, m = count_edges(out_path)
            print(f"    done ({n} nodes, {m} edges)")
            return name, n, m, name, out_path
    return None

def download_snap(name, url, datasets_dir):
    graph_dir = os.path.join(datasets_dir, name)
    os.makedirs(graph_dir, exist_ok=True)
    out_path = os.path.join(graph_dir, "refined_edges.txt")
    if os.path.exists(out_path):
        n, m = count_edges(out_path)
        print(f"  {name}: cached ({n} nodes, {m} edges)")
        return name, n, m, name, out_path
    print(f"  Downloading {name} (~200MB)...", end=" ", flush=True)
    try:
        gz_path = f"/tmp/{name}.txt.gz"
        urllib.request.urlretrieve(url, gz_path)
        print("unzipping...", end=" ", flush=True)
        with gzip.open(gz_path, "rt") as f_in, open(out_path, "w") as f_out:
            for line in f_in:
                f_out.write(line.replace("\t", " "))
        os.remove(gz_path)
        n, m = count_edges(out_path)
        print(f"done ({n} nodes, {m} edges)")
        return name, n, m, name, out_path
    except Exception as e:
        print(f"FAILED: {e}")
        return None

print("Loading soc-Pokec...")
result = get_local_graph("soc-Pokec", DRIVE_PATHS, datasets_dir)
if result is None:
    print("  Not in Drive, downloading from SNAP...")
    result = download_snap("soc-Pokec", "https://snap.stanford.edu/data/soc-pokec-relationships.txt.gz", datasets_dir)
if result:
    graph_info.append(result)
print(f"\nTotal: {len(graph_info)} graphs ready")

In [ ]:
# @title 5. Run benchmark

def run_and_parse(binary, graph, threads, method, timeout=600):
    result = subprocess.run([binary, graph, str(threads), str(method)],
                           capture_output=True, text=True, timeout=timeout)
    output = result.stdout + result.stderr
    runtime = scc = None
    for line in output.split('\n'):
        m = re.search(r'running_time\(ms\)=([0-9.]+)', line)
        if m: runtime = float(m.group(1))
        m = re.search(r'Total # SCCs = (\d+)', line)
        if m: scc = int(m.group(1))
    return runtime, scc, output

all_results = []
for name, n_nodes, m_edges, desc, graph_path in graph_info:
    print(f"\n{'='*70}")
    print(f"Graph: {name}  ({n_nodes} nodes, {m_edges} edges)")
    print(f"{'='*70}")
    row = {"name": name, "nodes": n_nodes, "edges": m_edges}
    for m in [0, 1, 2]:
        rt, scc, out = run_and_parse(BINARY_OMP, graph_path, 8, m)
        row[f"omp{m}_rt"] = rt; row[f"omp{m}_scc"] = scc
        print(f"  [OMP{m}] {f'{rt:.2f}ms' if rt else 'FAIL'}  SCC={scc}")
    for m in [0, 1, 2]:
        rt, scc, out = run_and_parse(BINARY_CUDA, graph_path, 8, m)
        row[f"cuda{m}_rt"] = rt; row[f"cuda{m}_scc"] = scc
        print(f"  [CUDA{m}] {f'{rt:.2f}ms' if rt else 'FAIL'}  SCC={scc}")
    all_results.append(row)
print("\n✓ All benchmarks complete")

In [ ]:
# @title 6. Benchmark Dashboard (OMP vs CUDA)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML
pd.set_option('display.max_colwidth', 30)

def safe(v): return v if (v is not None and v > 0) else None
def best_rt(r, prefix):
    vals = [safe(r.get(f'{prefix}{i}_rt')) for i in range(3)]
    vals = [v for v in vals if v is not None]
    return min(vals) if vals else None
def scc_val(r, prefix): return r.get(f'{prefix}2_scc')
def speedup(o, c): return round(o/c, 2) if (o and c and c > 0) else None

rows = []
for r in all_results:
    o, c = best_rt(r, 'omp'), best_rt(r, 'cuda')
    sp = speedup(o, c)
    oscc, cscc = scc_val(r, 'omp'), scc_val(r, 'cuda')
    match = (oscc == cscc) if (oscc is not None and cscc is not None) else False
    rows.append({
        'Graph': r['name'], 'Nodes': f"{r['nodes']:,}", 'Edges': f"{r['edges']:,}",
        'OMP(ms)': round(o, 2) if o else None, 'CUDA(ms)': round(c, 2) if c else None,
        'Speedup': sp, 'Winner': 'CUDA' if (sp and sp >= 1) else ('OMP' if (sp and sp < 1) else '—'),
        'SCC ✓': '✓' if match else '✗',
    })
df = pd.DataFrame(rows)
spds = [r['Speedup'] for r in rows if r['Speedup']]
avg_sp = np.mean(spds) if spds else 0; max_sp = max(spds) if spds else 0
cuda_w = sum(1 for r in rows if r['Winner'] == 'CUDA'); tot = len(rows)
all_ok = all(r['SCC ✓'] == '✓' for r in rows) if rows else False

display(HTML("""<style>
  .db { font-family: -apple-system, sans-serif; max-width: 1000px; margin: 0 auto; }
  .cards { display: flex; gap: 12px; margin-bottom: 20px; flex-wrap: wrap; }
  .card { flex: 1; min-width: 130px; border-radius: 10px; padding: 16px 18px; text-align: center; color: #fff; box-shadow: 0 3px 12px rgba(0,0,0,0.18); }
  .card .v { font-size: 26px; font-weight: 700; margin: 2px 0; }
  .card .l { font-size: 11px; opacity: .7; text-transform: uppercase; letter-spacing: 1px; }
  .card .s { font-size: 10px; opacity: .5; }
  .c1 { background: linear-gradient(135deg,#1a1a2e,#16213e); }
  .c2 { background: linear-gradient(135deg,#0f4c3a,#1b7a4a); }
  .c3 { background: linear-gradient(135deg,#5c3d0e,#a67c00); }
</style>"""))

if not all_results:
    display(HTML('<div class="db"><p style="padding:40px;text-align:center;color:#6b7280">Run Cells 4–5 first.</p></div>'))
else:
    display(HTML(f'''<div class="db"><div class="cards">
      <div class="card c1"><div class="l">Graphs</div><div class="v">{tot}</div><div class="s">{df["Nodes"].iloc[0]} max nodes</div></div>
      <div class="card c2"><div class="l">CUDA Wins</div><div class="v">{cuda_w}/{tot}</div><div class="s">faster on GPU</div></div>
      <div class="card c3"><div class="l">Avg Speedup</div><div class="v">{avg_sp:.1f}×</div><div class="s">max {max_sp:.1f}×</div></div>
      <div class="card c1"><div class="l">SCC Match</div><div class="v" style="color:{'#34d399' if all_ok else '#f87171'}">{'✓ All' if all_ok else '⚠'}</div><div class="s">correct</div></div>
    </div>"""))

    names = [r['Graph'] for r in rows]
    omp_t = [r['OMP(ms)'] or 0 for r in rows]
    cuda_t = [r['CUDA(ms)'] or 0 for r in rows]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5), gridspec_kw={'width_ratios': [2.2, 1]})
    x = np.arange(len(names)); w = 0.32
    b1 = ax1.bar(x - w/2, omp_t, w, label='OpenMP', color='#f59e0b', edgecolor='#d97706', lw=.7, alpha=.9)
    b2 = ax1.bar(x + w/2, cuda_t, w, label='CUDA', color='#3b82f6', edgecolor='#2563eb', lw=.7, alpha=.9)
    for b in b1: ax1.text(b.get_x()+b.get_width()/2, b.get_height()+max(omp_t+cuda_t)*.01, f'{b.get_height():.1f}', ha='center', fontsize=7, fontweight=600, color='#92400e')
    for b in b2: ax1.text(b.get_x()+b.get_width()/2, b.get_height()+max(omp_t+cuda_t)*.01, f'{b.get_height():.1f}', ha='center', fontsize=7, fontweight=600, color='#1e40af')
    ax1.set_xticks(x); ax1.set_xticklabels(names, fontsize=9)
    ax1.set_ylabel('Runtime (ms)', fontsize=10, fontweight=600)
    ax1.set_title('Runtime', fontsize=12, fontweight=700); ax1.legend(fontsize=8)
    ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False); ax1.grid(axis='y', alpha=.3)
    for i, r in enumerate(rows):
        sp = r['Speedup']
        if sp and sp >= .9:
            ax1.annotate(f'{sp:.1f}×', (x[i], max(omp_t[i],cuda_t[i])+max(omp_t+cuda_t)*.05), ha='center', fontsize=8, fontweight=700, color='#059669' if sp >= 1 else '#dc2626', bbox=dict(boxstyle='round,pad=.1', fc='white', ec='#d1d5db', alpha=.9))

    chart_spds = [r['Speedup'] or 0 for r in rows]
    clrs = ['#059669' if s >= 1 else '#dc2626' for s in chart_spds]
    ax2.barh(names, chart_spds, color=clrs, edgecolor='white', linewidth=1, height=.5)
    for i, s in enumerate(chart_spds):
        if s > 0: ax2.text(s + .03, i, f'{s:.1f}×', va='center', fontsize=10, fontweight=700, color='#059669' if s >= 1 else '#dc2626')
    ax2.axvline(1, color='#6b7280', ls='--', lw=.8, alpha=.6)
    ax2.set_xlabel('Speedup → CUDA better', fontsize=9, fontweight=600)
    ax2.set_title('Speedup', fontsize=12, fontweight=700)
    ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False); ax2.grid(axis='x', alpha=.3)
    ax2.set_xlim(left=0)
    plt.tight_layout(); plt.show()

    def c_time(v): return 'background:#f3f4f6;color:#9ca3af' if v is None else ''
    def c_sp(v):
        if v is None: return 'color:#9ca3af'
        if v >= 2: return 'color:#065f46;font-weight:700;font-size:14px'
        if v >= 1: return 'color:#059669;font-weight:600'
        return 'color:#dc2626;font-weight:600'
    def c_win(v):
        if v == 'CUDA': return 'background:#ecfdf5;color:#065f46;font-weight:600;text-align:center'
        if v == 'OMP': return 'background:#fffbeb;color:#92400e;font-weight:600;text-align:center'
        return 'text-align:center'
    def c_ok(v): return 'color:#059669;font-weight:700;font-size:16px;text-align:center' if v == '✓' else 'color:#dc2626;font-weight:700;text-align:center'

    styled = (df.style
        .format({'OMP(ms)': lambda v: f'{v:.1f}' if pd.notna(v) else 'FAIL',
                 'CUDA(ms)': lambda v: f'{v:.1f}' if pd.notna(v) else 'FAIL',
                 'Speedup': lambda v: f'{v:.1f}×' if pd.notna(v) else '—'})
        .applymap(c_time, subset=['OMP(ms)', 'CUDA(ms)'])
        .applymap(c_sp, subset=['Speedup'])
        .applymap(c_win, subset=['Winner'])
        .applymap(c_ok, subset=['SCC ✓'])
        .set_table_styles([
            {'selector': 'th', 'props': [('background','#1a1a2e'), ('color','white'), ('padding','9px 10px'), ('font-weight','600'), ('font-size','11px'), ('text-transform','uppercase'), ('letter-spacing','.5px')]},
            {'selector': 'td', 'props': [('padding','8px 10px'), ('border-bottom','1px solid #e5e7eb'), ('font-size','12px')]},
            {'selector': 'tr:hover td', 'props': [('background','#f9fafb')]},
        ])
        .set_caption("OMP vs CUDA — Best of Methods 0/1/2")
    )
    display(HTML('<div class="db"><h3>Comparison</h3><div class="sub">Best runtime · SCC from Method 2 (full pipeline)</div></div>'))
    display(styled)

    det = []
    for r in all_results:
        for m in range(3):
            o_rt, o_scc = safe(r.get(f'omp{m}_rt')), r.get(f'omp{m}_scc')
            c_rt, c_scc = safe(r.get(f'cuda{m}_rt')), r.get(f'cuda{m}_scc')
            mt = '✓' if (o_scc == c_scc) and (o_scc is not None) else ('✗' if o_scc is not None else '?')
            det.append({'Graph': r['name'], 'M': f'M{m}', 'OMP(ms)': round(o_rt,2) if o_rt else None, 'OMP-SCC': o_scc or '?', 'CUDA(ms)': round(c_rt,2) if c_rt else None, 'CUDA-SCC': c_scc or '?', 'Match': mt})
    det_df = pd.DataFrame(det)
    def c_det(v): return 'background:#f3f4f6;color:#9ca3af' if v is None else ''
    def c_m(v): return 'color:#059669;font-weight:700;text-align:center' if v == '✓' else ('color:#dc2626;font-weight:700;text-align:center' if v == '✗' else 'text-align:center;color:#9ca3af')
    det_styled = (det_df.style
        .format({'OMP(ms)': lambda v: f'{v:.1f}' if pd.notna(v) else 'FAIL', 'CUDA(ms)': lambda v: f'{v:.1f}' if pd.notna(v) else 'FAIL'})
        .applymap(c_det, subset=['OMP(ms)', 'CUDA(ms)'])
        .applymap(c_m, subset=['Match'])
        .set_table_styles([{'selector':'th','props':[('background','#1a1a2e'),('color','white'),('padding','7px 8px'),('font-weight','600'),('font-size','10px'),('text-transform','uppercase')]},
                          {'selector':'td','props':[('padding','6px 8px'),('border-bottom','1px solid #e5e7eb'),('font-size','11px')]},
                          {'selector':'tr:hover td','props':[('background','#f9fafb')]}])
        .set_caption("Per-Method Breakdown")
    )
    display(HTML('<div class="db" style="margin-top:8px"></div>'))
    display(det_styled)

In [ ]:
# @title 7. Export to Excel
import pandas as pd
from google.colab import files

rows = []
for r in all_results:
    def g(k): return r.get(k, -1)
    rows.append({
        "Graph": r["name"], "Nodes": r["nodes"], "Edges": r["edges"],
        "OMP_best_ms": min([g(f"omp{i}_rt") for i in range(3) if g(f"omp{i}_rt") > 0], default="FAIL"),
        "OMP_SCC": g("omp2_scc"),
        "CUDA_best_ms": min([g(f"cuda{i}_rt") for i in range(3) if g(f"cuda{i}_rt") > 0], default="FAIL"),
        "CUDA_SCC": g("cuda2_scc"),
        "OMP0_ms": g("omp0_rt"), "OMP1_ms": g("omp1_rt"), "OMP2_ms": g("omp2_rt"),
        "CUDA0_ms": g("cuda0_rt"), "CUDA1_ms": g("cuda1_rt"), "CUDA2_ms": g("cuda2_rt"),
    })
df = pd.DataFrame(rows)
with pd.ExcelWriter("/content/scc_benchmark_results.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Comparison", index=False)
print("✓ Excel exported")
files.download("/content/scc_benchmark_results.xlsx")